In [ ]:
%pip install requests tqdm


In [ ]:
import os
import requests
from datetime import datetime, timedelta
from tqdm import tqdm


In [ ]:
def daterange(start_date, end_date):
    for n in range((end_date - start_date).days + 1):
        yield start_date + timedelta(days=n)


In [ ]:
def download_modis(date, save_dir, collection="61"):
    date_str = date.strftime("%Y/%m/%d")
    url = f"https://ladsweb.modaps.eosdis.nasa.gov/archive/allData/{collection}/MYD06_L2/{date_str}/"

    token = os.environ.get("EARTHDATA_TOKEN")

    if not token:
        raise ValueError("EARTHDATA_TOKEN environment variable is not set")

    headers = {"Authorization": f"Bearer {token}"}

    response = requests.get(url, headers=headers, timeout=60)

    if response.status_code == 200:
        files = [
            line.split('\"')[1]
            for line in response.text.split('\n')
            if 'MYD06_L2.A' in line and '.hdf' in line
        ]

        for file in files:
            file_url = url + file
            out_path = os.path.join(save_dir, file)

            if os.path.exists(out_path):
                continue

            print(f"Downloading: {file}")

            with requests.get(
                file_url,
                headers=headers,
                stream=True,
                timeout=120
            ) as file_response:

                file_response.raise_for_status()

                with open(out_path, "wb") as output_file:
                    for chunk in file_response.iter_content(chunk_size=8192):
                        if chunk:
                            output_file.write(chunk)

    else:
        print(
            f"Failed to get file list for {date_str}. "
            f"Status code: {response.status_code}"
        )


In [ ]:
start_date = datetime(2002, 7, 4)
end_date = datetime(2008, 12, 31)

save_dir = "./MODIS_AQUA"
os.makedirs(save_dir, exist_ok=True)


In [ ]:
for single_date in tqdm(daterange(start_date, end_date)):
    download_modis(single_date, save_dir)
